## 0. Установка зависимостей


In [81]:
!pip install -q pandas numpy scipy requests beautifulsoup4 plotly "dash>=2.11" pytrends


INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1779305972.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1779305972.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1779305972.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1779305972.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m1779305972.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:54:00] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/May/2026 19:

## 1. Импорты


In [82]:
import io
import json
import math
import os
import re
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
from scipy import stats

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)


## 2. База


In [83]:
@dataclass
class ModuleResult:
    name: str
    tables: Dict[str, pd.DataFrame] = field(default_factory=dict)
    figures: Dict[str, Any] = field(default_factory=dict)
    findings: List[str] = field(default_factory=list)

    def add_finding(self, text: str) -> None:
        self.findings.append(text.strip())


class ResearchModule(ABC):
    def __init__(self, name: str, data_dir: str = DATA_DIR) -> None:
        self.name = name
        self.data_dir = data_dir
        os.makedirs(self.data_dir, exist_ok=True)
        self.result = ModuleResult(name=name)

    @abstractmethod
    def fetch(self) -> None: ...

    @abstractmethod
    def transform(self) -> None: ...

    @abstractmethod
    def visualize(self) -> None: ...

    def run(self) -> ModuleResult:
        self.fetch()
        self.transform()
        self.visualize()
        return self.result


## 3. Оценка выборок и гипотезы


In [84]:
@dataclass
class SampleSummary:
    n: int
    mean: float
    median: float
    std: float
    ci_low: float
    ci_high: float


class SampleEvaluator:
    def __init__(self, alpha: float = 0.05) -> None:
        self.alpha = alpha

    def summarize(self, values: Sequence[float]) -> SampleSummary:
        arr = np.asarray(list(values), dtype=float)
        arr = arr[~np.isnan(arr)]
        n = arr.size
        if n == 0:
            return SampleSummary(0, np.nan, np.nan, np.nan, np.nan, np.nan)
        mean = float(arr.mean())
        median = float(np.median(arr))
        std = float(arr.std(ddof=1)) if n > 1 else 0.0
        if n > 1:
            t_crit = stats.t.ppf(1 - self.alpha / 2, df=n - 1)
            margin = t_crit * std / np.sqrt(n)
        else:
            margin = 0.0
        return SampleSummary(n=n, mean=mean, median=median, std=std,
                             ci_low=mean - margin, ci_high=mean + margin)

    def remove_outliers_iqr(self, values: Iterable[float]) -> np.ndarray:
        arr = np.asarray(list(values), dtype=float)
        if arr.size < 4:
            return arr
        q1, q3 = np.percentile(arr, [25, 75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return arr[(arr >= lo) & (arr <= hi)]


class HypothesisTester:
    def __init__(self, alpha: float = 0.05) -> None:
        self.alpha = alpha

    def two_samples_diff(self, sample_a, sample_b, equal_var: bool = False):
        a = np.asarray(list(sample_a), dtype=float)
        b = np.asarray(list(sample_b), dtype=float)
        a = a[~np.isnan(a)]
        b = b[~np.isnan(b)]
        if a.size < 2 or b.size < 2:
            return 0.0, 1.0, False
        t_stat, p_value = stats.ttest_ind(a, b, equal_var=equal_var)
        return float(t_stat), float(p_value), bool(p_value < self.alpha)


## 4. Хелперы для графиков (попросил нейронку тут помочь сделать красиво)


In [85]:
PALETTE = ["#5B6CFF", "#FF7E7E", "#33C7A4", "#FFB547", "#9B6BFF", "#3B98FF"]


def _layout(fig: go.Figure, title: str) -> go.Figure:
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=18)),
        template="plotly_white",
        margin=dict(l=40, r=20, t=60, b=40),
        colorway=PALETTE,
        font=dict(family="Inter, Arial, sans-serif", size=13),
    )
    return fig


def build_bar(df, x, y, title, color=None, orientation="v"):
    fig = px.bar(df, x=x, y=y, color=color, orientation=orientation, text_auto=".2s")
    fig.update_traces(textposition="outside", cliponaxis=False)
    return _layout(fig, title)


def build_box(df, x, y, title):
    fig = px.box(df, x=x, y=y, points="suspectedoutliers")
    return _layout(fig, title)


def build_scatter(df, x, y, title, color=None, size=None, text=None):
    fig = px.scatter(df, x=x, y=y, color=color, size=size, text=text)
    fig.update_traces(textposition="top center")
    return _layout(fig, title)


def build_line(df, x, y, title, color=None):
    fig = px.line(df, x=x, y=y, color=color, markers=True)
    return _layout(fig, title)


## 5. Анализ контрагентов


In [86]:
WIKI_COMPANIES = [
    "Skillbox",
    "GeekBrains",
    "Нетология",
    "Skyeng",
    "Stepik",
    "Учи.ру",
    "Яндекс_Практикум",
    "Фоксфорд",
    "Maximum_Education",
    "Skillfactory",
    "Яндекс_Учебник",
]


class WikipediaCompanyParser:
    BASE = "https://ru.wikipedia.org/wiki/"
    HEADERS = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

    LABEL_MAP = {
        "основан": "founded",
        "основание": "founded",
        "дата основания": "founded",
        "год основания": "founded",
        "расположение": "headquarters",
        "штаб-квартира": "headquarters",
        "местоположение": "headquarters",
        "число сотрудников": "employees",
        "сотрудников": "employees",
        "отрасль": "industry",
        "тип": "type",
        "выручка": "revenue",
        "владельцы": "owners",
        "ключевые фигуры": "key_people",
        "материнская компания": "parent",
        "сайт": "website",
    }

    def __init__(self, name: str) -> None:
        self.name = name

    def _norm_text(self, value: str) -> str:
        value = re.sub(r"\[\d+\]", "", value)
        return " ".join(value.split())

    def parse(self) -> Dict[str, Any]:
        url = self.BASE + self.name
        response = requests.get(url, headers=self.HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        result: Dict[str, Any] = {"name": self.name.replace("_", " "), "wiki_url": url}
        infobox = soup.select_one("table.infobox")
        if not infobox:
            return result

        for row in infobox.select("tr"):
            head = row.select_one("th")
            cell = row.select_one("td")
            if not head or not cell:
                continue
            label_raw = self._norm_text(head.get_text(" ", strip=True)).lower()
            key = None
            for prefix, target in self.LABEL_MAP.items():
                if label_raw.startswith(prefix):
                    key = target
                    break
            if not key:
                continue
            result[key] = self._norm_text(cell.get_text(" ", strip=True))
        return result


def parse_wikipedia_companies(names: List[str]) -> pd.DataFrame:
    rows = []
    for slug in names:
        try:
            data = WikipediaCompanyParser(slug).parse()
            rows.append(data)
            time.sleep(0.3)
        except Exception as exc:
            print(f"Не удалось обработать {slug}: {exc}")
    return pd.DataFrame(rows)


def extract_year(value: Any) -> Optional[int]:
    if not isinstance(value, str):
        return None
    match = re.search(r"(19|20)\d{2}", value)
    return int(match.group()) if match else None


def extract_headcount(value: Any) -> Optional[int]:
    if not isinstance(value, str):
        return None
    digits = re.sub(r"[^\d]", "", value.split("(")[0])
    return int(digits) if digits else None


`WikipediaCompanyParser` шоб я не забыл - это статический парсер, вроде реализовал


In [87]:
class ContractorsResearch(ResearchModule):
    def __init__(self, company_names: List[str] = WIKI_COMPANIES) -> None:
        super().__init__(name="contractors")
        self.company_names = company_names
        self.evaluator = SampleEvaluator()
        self.tester = HypothesisTester()
        self.raw: Optional[pd.DataFrame] = None

    def fetch(self) -> None:
        self.raw = parse_wikipedia_companies(self.company_names)

    def transform(self) -> None:
        df = self.raw.copy()
        df["founded_year"] = df.get("founded", pd.Series(dtype=object)).apply(extract_year)
        df["employees_count"] = df.get("employees", pd.Series(dtype=object)).apply(extract_headcount)

        df["is_moscow"] = df.get("headquarters", "").fillna("").str.contains("Москва", case=False)
        df["age_years"] = df["founded_year"].apply(
            lambda y: 2026 - y if isinstance(y, int) else None
        )

        location_summary = (
            df.assign(city=df.get("headquarters", "").fillna("не указано"))
            ["city"].str.extract(r"(Москва|Санкт-Петербург|[А-ЯЁ][а-яё]+)")[0]
            .value_counts()
            .reset_index()
        )
        location_summary.columns = ["city", "companies"]

        decade_summary = (
            df.dropna(subset=["founded_year"])
            .assign(decade=lambda x: (x["founded_year"] // 10 * 10).astype(int))
            .groupby("decade")
            .size()
            .reset_index(name="companies")
            .sort_values("decade")
        )

        self.result.tables["companies"] = df
        self.result.tables["locations"] = location_summary
        self.result.tables["decades"] = decade_summary

    def visualize(self) -> None:
        df = self.result.tables["companies"]
        locations = self.result.tables["locations"]
        decades = self.result.tables["decades"]

        self.result.figures["companies_by_city"] = build_bar(
            locations, x="city", y="companies",
            title="Распределение EdTech - подрядчиков по городам (Wikipedia)",
        )
        self.result.figures["founding_decade"] = build_bar(
            decades, x="decade", y="companies",
            title="Год основания подрядчиков (десятилетие)",
        )
        plot_df = df.dropna(subset=["employees_count"]).sort_values("employees_count", ascending=False)
        if not plot_df.empty:
            self.result.figures["employees_by_company"] = build_bar(
                plot_df.head(10), x="name", y="employees_count",
                title="Численность штата по данным Wikipedia",
            )

        msk_share = float(df["is_moscow"].mean()) * 100
        years = df["age_years"].dropna()
        age_summary = self.evaluator.summarize(years)

        self.result.add_finding(
            f"Из {len(df)} разобранных EdTech-компаний {msk_share:.0f} процентов базируются в Москве. "
            "Для нас это значит, что переговоры по платформам и подрядчикам можно вести из Москвы "
            "без командировок."
        )
        if age_summary.n > 0:
            self.result.add_finding(
                f"Средний возраст подрядчика {age_summary.mean:.1f} лет (CI: "
                f"{age_summary.ci_low:.1f} - {age_summary.ci_high:.1f}). "
                "Рынок зрелый, есть из чего выбирать."
            )
        big_players = df.dropna(subset=["employees_count"]).query("employees_count >= 500")
        if not big_players.empty:
            self.result.add_finding(
                f"Крупных игроков со штатом 500+ человек: {len(big_players)} "
                f"({', '.join(big_players['name'].head(5).tolist())}). "
                "С такими работать выгодно за счёт инфраструктуры, но дорого по интеграции."
            )


In [88]:
contractors_result = ContractorsResearch().run()
display(contractors_result.tables["companies"].head(10))

for figure in contractors_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in contractors_result.findings:
    print(" -", item)


,name,wiki_url,type,founded,headquarters,key_people,industry,parent,website,employees,founded_year,employees_count,is_moscow,age_years
0,Skillbox,https://ru.wikipedia.org/wiki/Skillbox,общество с ограниченной ответственностью,2016,Россия : Москва,Артём Казаков (генеральный директор) [ 2 ],образование ( МСОК : 85 ),Skillbox Holding ( VK ),skillbox.ru,NaN,2016.0,NaN,True,None
1,GeekBrains,https://ru.wikipedia.org/wiki/GeekBrains,Общество с ограниченной ответственностью,2010,Россия : Москва,Дмитрий Крутов — генеральный директор [ 1 ],образование ( МСОК : 85 ),Skillbox Holding ( VK ),gb.ru,▲ 568 (2021) [ 2 ],2010.0,568.0,True,None
2,Нетология,https://ru.wikipedia.org/wiki/Нетология,частная компания,2011,Россия : Москва,Марианна Снигирева (генеральный директор) Алек...,онлайн-образование,NaN,netology.ru,1500 (2022),2011.0,1500.0,True,None
3,Skyeng,https://ru.wikipedia.org/wiki/Skyeng,частная компания,2012,Россия : Москва,Александр Ларьяновский (управляющий партнёр),образование,NaN,skyeng.ru,NaN,2012.0,NaN,True,None
4,Stepik,https://ru.wikipedia.org/wiki/Stepik,Онлайн-образование,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,None
5,Учи.ру,https://ru.wikipedia.org/wiki/Учи.ру,образовательная платформа,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,None
6,Яндекс Практикум,https://ru.wikipedia.org/wiki/Яндекс_Практикум,Дочернее предприятие,2019,Россия : Москва,Илья Курмышев — генеральный директор,Образование,Яндекс,practicum.yandex.ru,NaN,2019.0,NaN,True,None
7,Фоксфорд,https://ru.wikipedia.org/wiki/Фоксфорд,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,None
8,Maximum Education,https://ru.wikipedia.org/wiki/Maximum_Education,организация,2013,"Москва , Россия",NaN,NaN,NaN,maximumtest.ru (рус.),NaN,2013.0,NaN,True,None
9,Skillfactory,https://ru.wikipedia.org/wiki/Skillfactory,Частная компания,2016,Россия : Москва,"Злыгостева Мария (генеральный директор), Суноз...",образование ( МСОК : 85 ),Skillbox Holding ( VK ),skillfactory.ru,NaN,2016.0,NaN,True,None


Выводы по разделу:
 - Из 11 разобранных EdTech-компаний 64 процентов базируются в Москве. Для нас это значит, что переговоры по платформам и подрядчикам можно вести из Москвы без командировок.
 - Крупных игроков со штатом 500+ человек: 2 (GeekBrains, Нетология). С такими работать выгодно за счёт инфраструктуры, но дорого по интеграции.


## 6. География компании



In [ ]:
CITIES_WIKI_URL = (
    "https://ru.wikipedia.org/wiki/"
    "Список_городов_России_с_населением_более_100_тысяч_человек"
)
HEADERS_WIKI = {"User-Agent": "GlowUpResearch/1.0 (course project)"}


STUDENT_SHARE = 0.058
TOP_CITIES = 15


def load_students_dataset() -> pd.DataFrame:
    response = requests.get(CITIES_WIKI_URL, headers=HEADERS_WIKI, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.select_one("table.wikitable")
    if table is None:
        raise RuntimeError("Не нашёл таблицу с населением в статье Wikipedia")

    rows = []
    for tr in table.select("tr")[1:]:
        cells = [td.get_text(" ", strip=True) for td in tr.select("td")]
        if len(cells) < 4:
            continue
        city = re.sub(r"\[\d+\]", "", cells[2]).strip()
        last_value = re.sub(r"[^\d]", "", cells[-1])
        if not last_value:
            continue
        population_thousand = int(last_value)
        rows.append({"city": city, "population_thousand": population_thousand})

    df = pd.DataFrame(rows).drop_duplicates(subset="city")
    df = df.sort_values("population_thousand", ascending=False).head(TOP_CITIES).reset_index(drop=True)
    df["students_thousand"] = (df["population_thousand"] * STUDENT_SHARE).round(0)
    return df


class CityGeocoder:
    ENDPOINT = "https://nominatim.openstreetmap.org/search"
    HEADERS = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

    def __init__(self) -> None:
        self.cache: Dict[str, Tuple[float, float]] = {}

    def geocode(self, city: str) -> Optional[Tuple[float, float]]:
        if city in self.cache:
            return self.cache[city]
        params = {"q": f"{city}, Россия", "format": "json", "limit": 1}
        try:
            response = requests.get(self.ENDPOINT, params=params,
                                    headers=self.HEADERS, timeout=15)
            response.raise_for_status()
            data = response.json()
            if not data:
                return None
            point = (float(data[0]["lat"]), float(data[0]["lon"]))
            self.cache[city] = point
            time.sleep(1.1)
            return point
        except Exception as exc:
            print(f"Nominatim не ответил для {city}: {exc}")
            return None


def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def estimate_delivery_cost(distance_km: float, weight_kg: float = 0.6) -> float:
    return 250 + 0.45 * distance_km + 80 * weight_kg


In [ ]:
class GeographyResearch(ResearchModule):
    STUDIO_CITY = "Москва"

    def __init__(self) -> None:
        super().__init__(name="geography")
        self.evaluator = SampleEvaluator()
        self.cities: Optional[pd.DataFrame] = None
        self.geocoder = CityGeocoder()

    def fetch(self) -> None:
        df = load_students_dataset()
        coords = []
        for city in df["city"]:
            point = self.geocoder.geocode(city)
            if point is None:
                point = (None, None)
            coords.append(point)
        df[["lat", "lon"]] = pd.DataFrame(coords, index=df.index)
        df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
        self.cities = df

    def transform(self) -> None:
        df = self.cities.copy()
        df["potential_customers_thousand"] = (df["students_thousand"] * 0.45).round(1)

        studio = df.loc[df["city"] == self.STUDIO_CITY]
        if not studio.empty:
            sx = studio.iloc[0]
            df["distance_from_studio_km"] = df.apply(
                lambda row: haversine_km(sx["lat"], sx["lon"], row["lat"], row["lon"]),
                axis=1,
            ).round(0)
            df["delivery_cost_rub"] = df["distance_from_studio_km"].apply(estimate_delivery_cost).round(0)

        df = df.sort_values("potential_customers_thousand", ascending=False).reset_index(drop=True)
        self.result.tables["cities"] = df

    def visualize(self) -> None:
        df = self.result.tables["cities"]
        top10 = df.head(10)

        self.result.figures["potential_audience"] = build_bar(
            top10, x="city", y="potential_customers_thousand",
            title="Потенциальная аудитория по городам, тыс. человек",
        )
        if "delivery_cost_rub" in df.columns:
            self.result.figures["delivery_cost"] = build_bar(
                top10, x="city", y="delivery_cost_rub",
                title="Стоимость отправки welcome-бокса из Москвы, руб.",
            )
            self.result.figures["audience_vs_delivery"] = build_scatter(
                df, x="delivery_cost_rub", y="potential_customers_thousand",
                title="Аудитория vs логистика", text="city", size="students_thousand",
            )

        msk = float(df.loc[df["city"] == "Москва", "potential_customers_thousand"].iloc[0])
        spb = float(df.loc[df["city"] == "Санкт-Петербург", "potential_customers_thousand"].iloc[0])
        total = float(df["potential_customers_thousand"].sum())
        cap_share = (msk + spb) / total * 100

        self.result.add_finding(
            f"На две столицы приходится {cap_share:.0f} процентов потенциальной аудитории. "
            "Студию держим в Москве, дополнительно ездим в Петербург."
        )
        if "delivery_cost_rub" in df.columns:
            avg_delivery = self.evaluator.summarize(df["delivery_cost_rub"])
            self.result.add_finding(
                f"Средняя стоимость отправки welcome-бокса из Москвы: {avg_delivery.mean:.0f} руб. "
                f"Доверительный интервал: {avg_delivery.ci_low:.0f} - {avg_delivery.ci_high:.0f} руб."
            )


In [91]:
geography_result = GeographyResearch().run()
display(geography_result.tables["cities"].head(10))

for figure in geography_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in geography_result.findings:
    print(" -", item)


,city,population_thousand,students_thousand,lat,lon,potential_customers_thousand,distance_from_studio_km,delivery_cost_rub
0,Москва,13274,770.0,55.625578,37.606392,346.5,0.0,298.0
1,Санкт-Петербург,5653,328.0,59.960674,30.158655,147.6,653.0,592.0
2,Новосибирск,1637,95.0,55.028831,82.922689,42.8,2816.0,1565.0
3,Екатеринбург,1548,90.0,56.838207,60.600789,40.5,1421.0,937.0
4,Казань,1330,77.0,55.794649,49.111502,34.6,720.0,622.0
5,Красноярск,1212,70.0,56.009117,92.872586,31.5,3359.0,1810.0
6,Нижний Новгород,1198,69.0,56.326482,44.005139,31.0,406.0,481.0
7,Челябинск,1177,68.0,55.159841,61.402555,30.6,1496.0,971.0
8,Уфа,1166,68.0,54.726141,55.947499,30.6,1165.0,822.0
9,Краснодар,1155,67.0,45.035153,38.977240,30.2,1182.0,830.0


Выводы по разделу:
 - На две столицы приходится 54 процентов потенциальной аудитории. Студию держим в Москве, дополнительно ездим в Петербург.
 - Средняя стоимость отправки welcome-бокса из Москвы: 864 руб. Доверительный интервал: 635 - 1093 руб.


## 7. Анализ сотрудников



In [ ]:
ROLE_QUERIES = {
    "stylist": "стилист имиджмейкер",
    "cosmetologist": "косметолог эстетист",
    "methodologist": "методолог онлайн курс",
    "video_editor": "видеомонтажер reels",
    "smm": "smm менеджер онлайн школа",
    "support": "куратор онлайн школа",
}

BROWSER_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)


class HHLiveFetcher:
    API_ENDPOINT = "https://api.hh.ru/vacancies"
    WEB_ENDPOINT = "https://hh.ru/search/vacancy"

    def __init__(self, area: int = 113, per_query: int = 50) -> None:
        self.area = area
        self.per_query = per_query
        self.api_headers = {"User-Agent": "GlowUpResearch/1.0 (course project)"}
        self.web_headers = {"User-Agent": BROWSER_UA}

    def _from_api(self, query: str) -> List[Dict[str, Any]]:
        params = {
            "text": query,
            "area": self.area,
            "per_page": self.per_query,
            "only_with_salary": "true",
        }
        response = requests.get(self.API_ENDPOINT, params=params,
                                headers=self.api_headers, timeout=20)
        response.raise_for_status()
        payload = response.json()
        rows = []
        for raw in payload.get("items", []):
            salary = raw.get("salary") or {}
            if salary.get("currency") not in (None, "RUR"):
                continue
            rows.append({
                "name": raw.get("name", ""),
                "city": (raw.get("area") or {}).get("name", ""),
                "salary_from": salary.get("from"),
                "salary_to": salary.get("to"),
                "experience": (raw.get("experience") or {}).get("name", ""),
                "employment": (raw.get("employment") or {}).get("name", ""),
            })
        return rows

    def _from_web(self, query: str) -> List[Dict[str, Any]]:
        params = {"text": query, "area": self.area, "only_with_salary": "true"}
        response = requests.get(self.WEB_ENDPOINT, params=params,
                                headers=self.web_headers, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        rows = []
        for card in soup.select('[data-qa="vacancy-serp__vacancy"]'):
            name_el = card.select_one('[data-qa="serp-item__title"]')
            salary_el = card.select_one('[data-qa="vacancy-serp__vacancy-compensation"]')
            employer_el = card.select_one('[data-qa="vacancy-serp__vacancy-employer"]')
            city_el = card.select_one('[data-qa="vacancy-serp__vacancy-address"]')
            if not name_el or not salary_el:
                continue
            sf, st = self._parse_salary(salary_el.get_text(" ", strip=True))
            rows.append({
                "name": name_el.get_text(" ", strip=True),
                "city": city_el.get_text(" ", strip=True) if city_el else "",
                "salary_from": sf,
                "salary_to": st,
                "experience": "",
                "employment": "",
            })
        return rows

    @staticmethod
    def _parse_salary(text: str) -> Tuple[Optional[int], Optional[int]]:
        digits = re.findall(r"\d[\d\s\u00a0]*", text)
        if not digits:
            return None, None
        nums = [int(d.replace(" ", "").replace("\u00a0", "")) for d in digits]
        if "от" in text.lower() and len(nums) == 1:
            return nums[0], None
        if "до" in text.lower() and len(nums) == 1:
            return None, nums[0]
        if len(nums) >= 2:
            return nums[0], nums[1]
        return nums[0], nums[0]

    def fetch_role(self, query: str) -> List[Dict[str, Any]]:
        try:
            rows = self._from_api(query)
            if rows:
                return rows
        except Exception as exc:
            print(f"hh.ru API недоступен ({exc}), пробуем веб-парсинг.")
        try:
            return self._from_web(query)
        except Exception as exc:
            print(f"hh.ru веб тоже недоступен для {query}: {exc}")
            return []


In [93]:
class EmployeesResearch(ResearchModule):
    def __init__(self) -> None:
        super().__init__(name="employees")
        self.evaluator = SampleEvaluator()
        self.tester = HypothesisTester()
        self.fetcher = HHLiveFetcher()

    def fetch(self) -> None:
        rows = []
        for role, query in ROLE_QUERIES.items():
            try:
                fetched = self.fetcher.fetch_role(query)
                for item in fetched:
                    rows.append({"role_group": role, **item})
                time.sleep(0.4)
            except Exception as exc:
                print(f"hh.ru не ответил для {role}: {exc}")
        self.vacancies = pd.DataFrame(rows)
        print(f"Получено вакансий: {len(self.vacancies)}")

    def transform(self) -> None:
        df = self.vacancies.copy()
        if df.empty:
            print("hh.ru не вернул вакансии. хз пока шо делать")
            self.result.tables["vacancies"] = df
            self.result.tables["salary_summary"] = pd.DataFrame(columns=["role_group", "vacancies", "avg_salary", "median_salary"])
            return

        df["salary_from"] = pd.to_numeric(df["salary_from"], errors="coerce")
        df["salary_to"] = pd.to_numeric(df["salary_to"], errors="coerce")
        df["salary_avg"] = df[["salary_from", "salary_to"]].mean(axis=1)
        df = df.dropna(subset=["salary_avg"])

        cleaned_parts = []
        for role, sub in df.groupby("role_group"):
            cleaned_values = self.evaluator.remove_outliers_iqr(sub["salary_avg"].tolist())
            cleaned_parts.append(sub[sub["salary_avg"].isin(cleaned_values)])
        df = pd.concat(cleaned_parts, ignore_index=True)

        salary_summary = (
            df.groupby("role_group")
            .agg(vacancies=("name", "count"),
                 avg_salary=("salary_avg", "mean"),
                 median_salary=("salary_avg", "median"))
            .reset_index()
            .sort_values("avg_salary", ascending=False)
        )

        self.result.tables["vacancies"] = df
        self.result.tables["salary_summary"] = salary_summary

    def visualize(self) -> None:
        salary_summary = self.result.tables["salary_summary"]
        df = self.result.tables["vacancies"]

        if salary_summary.empty:
            self.result.add_finding(
                "hh.ru API заблокировал текущий IP. Пока хз шо делать "
            )
            return

        self.result.figures["avg_salary_by_role"] = build_bar(
            salary_summary, x="role_group", y="avg_salary",
            title="Средняя зарплата по вакансиям hh.ru (live), руб./мес.",
        )
        self.result.figures["salary_box"] = build_box(
            df, x="role_group", y="salary_avg",
            title="Разброс зарплат внутри ролей (hh.ru live)",
        )

        msk = df.loc[df["city"] == "Москва", "salary_avg"]
        rest = df.loc[df["city"] != "Москва", "salary_avg"]
        if msk.size >= 2 and rest.size >= 2:
            t_stat, p_value, reject = self.tester.two_samples_diff(msk, rest)
            msk_sum = self.evaluator.summarize(msk)
            rest_sum = self.evaluator.summarize(rest)
            self.result.add_finding(
                "Сравнение Москвы и регионов по зарплатам (live hh.ru): t = {:.2f}, p = {:.3f}. "
                "Москва: {:.0f} руб., регионы: {:.0f} руб. {}".format(
                    t_stat, p_value, msk_sum.mean, rest_sum.mean,
                    "Разница значима." if reject else "Разница не значима."
                )
            )

        best_role = salary_summary.iloc[0]
        worst_role = salary_summary.iloc[-1]
        self.result.add_finding(
            f"Самая дорогая роль на рынке прямо сейчас: {best_role['role_group']} "
            f"({best_role['avg_salary']:.0f} руб./мес. в среднем по {best_role['vacancies']:.0f} вакансиям). "
            f"Самая дешёвая: {worst_role['role_group']} ({worst_role['avg_salary']:.0f} руб.)."
        )
        self.result.add_finding(
            f"Всего проанализировано {len(df)} реальных вакансий с hh.ru, "
            "выбросы убраны по IQR внутри каждой роли."
        )


In [74]:
employees_result = EmployeesResearch().run()
display(employees_result.tables["salary_summary"].round(0))

for figure in employees_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in employees_result.findings:
    print(" -", item)


hh.ru API недоступен (403 Client Error: Forbidden for url: https://api.hh.ru/vacancies?text=%D1%81%D1%82%D0%B8%D0%BB%D0%B8%D1%81%D1%82+%D0%B8%D0%BC%D0%B8%D0%B4%D0%B6%D0%BC%D0%B5%D0%B9%D0%BA%D0%B5%D1%80&area=113&per_page=50&only_with_salary=true), пробуем веб-парсинг.
hh.ru API недоступен (403 Client Error: Forbidden for url: https://api.hh.ru/vacancies?text=%D0%BA%D0%BE%D1%81%D0%BC%D0%B5%D1%82%D0%BE%D0%BB%D0%BE%D0%B3+%D1%8D%D1%81%D1%82%D0%B5%D1%82%D0%B8%D1%81%D1%82&area=113&per_page=50&only_with_salary=true), пробуем веб-парсинг.
hh.ru API недоступен (403 Client Error: Forbidden for url: https://api.hh.ru/vacancies?text=%D0%BC%D0%B5%D1%82%D0%BE%D0%B4%D0%BE%D0%BB%D0%BE%D0%B3+%D0%BE%D0%BD%D0%BB%D0%B0%D0%B9%D0%BD+%D0%BA%D1%83%D1%80%D1%81&area=113&per_page=50&only_with_salary=true), пробуем веб-парсинг.
hh.ru API недоступен (403 Client Error: Forbidden for url: https://api.hh.ru/vacancies?text=%D0%B2%D0%B8%D0%B4%D0%B5%D0%BE%D0%BC%D0%BE%D0%BD%D1%82%D0%B0%D0%B6%D0%B5%D1%80+reels&area=113&per

,role_group,vacancies,avg_salary,median_salary


Выводы по разделу:
 - hh.ru API заблокировал текущий IP. Пока хз шо делать


## 8. Маркетинговая кампания


In [75]:
from pytrends.request import TrendReq

KEYWORDS_RU = [
    "уход за лицом",
    "базовый гардероб",
    "looksmaxxing",
    "стилист онлайн",
]


class GoogleTrendsFetcher:
    def __init__(self, geo: str = "RU", timeframe: str = "today 12-m") -> None:
        self.pytrends = TrendReq(hl="ru-RU", tz=180)
        self.geo = geo
        self.timeframe = timeframe

    def interest(self, keywords: List[str]) -> pd.DataFrame:
        self.pytrends.build_payload(kw_list=keywords, geo=self.geo, timeframe=self.timeframe)
        df = self.pytrends.interest_over_time()
        if df.empty:
            return df
        if "isPartial" in df.columns:
            df = df.drop(columns=["isPartial"])
        return df.reset_index()

    def related_queries(self, keyword: str) -> pd.DataFrame:
        self.pytrends.build_payload(kw_list=[keyword], geo=self.geo, timeframe=self.timeframe)
        data = self.pytrends.related_queries().get(keyword) or {}
        top = data.get("top")
        return top if isinstance(top, pd.DataFrame) else pd.DataFrame(columns=["query", "value"])


In [76]:
class EdTechMarketParser:
    URL = "https://ru.wikipedia.org/wiki/Образовательные_технологии"
    HEADERS = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

    def fetch_text(self) -> str:
        response = requests.get(self.URL, headers=self.HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        article = soup.select_one("div.mw-parser-output")
        return article.get_text(" ", strip=True) if article else ""

    def extract_market_facts(self, text: str) -> pd.DataFrame:
        pattern = re.compile(
            r"([^.\n]*?(\d{1,3}(?:[\s\u00a0]\d{3})*|\d+[\.,]?\d*)\s*(млрд|миллиардов|млн|миллионов|трлн)[^.\n]*)",
            re.IGNORECASE,
        )
        rows = []
        for match in pattern.finditer(text):
            sentence = re.sub(r"\s+", " ", match.group(1)).strip()
            if any(k in sentence.lower() for k in ["рынок", "edtech", "образования", "образования.", "оборот"]):
                rows.append({"snippet": sentence[:240]})
            if len(rows) >= 8:
                break
        return pd.DataFrame(rows)


In [77]:
class MarketingResearch(ResearchModule):
    KEYWORDS = KEYWORDS_RU

    def __init__(self, average_check: float = 24_900) -> None:
        super().__init__(name="marketing")
        self.average_check = average_check
        self.trends_fetcher = GoogleTrendsFetcher()
        self.market_parser = EdTechMarketParser()
        self.evaluator = SampleEvaluator()
        self.trends_df: Optional[pd.DataFrame] = None
        self.market_df: Optional[pd.DataFrame] = None

    def fetch(self) -> None:
        try:
            self.trends_df = self.trends_fetcher.interest(self.KEYWORDS)
        except Exception as exc:
            print("Google Trends недоступен:", exc)
            self.trends_df = pd.DataFrame()

        try:
            text = self.market_parser.fetch_text()
            self.market_df = self.market_parser.extract_market_facts(text)
        except Exception as exc:
            print("Wikipedia EdTech статья недоступна:", exc)
            self.market_df = pd.DataFrame()

    def transform(self) -> None:
        if not self.trends_df.empty:
            tidy = self.trends_df.melt(id_vars="date", var_name="keyword", value_name="interest")
            trend_summary = (
                tidy.groupby("keyword")
                .agg(avg_interest=("interest", "mean"),
                     peak_interest=("interest", "max"),
                     last_value=("interest", "last"))
                .reset_index()
                .sort_values("avg_interest", ascending=False)
            )
            self.result.tables["trends_long"] = tidy
            self.result.tables["trends_summary"] = trend_summary

        if not self.market_df.empty:
            self.result.tables["market_facts"] = self.market_df

    def visualize(self) -> None:
        if "trends_long" in self.result.tables:
            tidy = self.result.tables["trends_long"]
            self.result.figures["trends_dynamics"] = build_line(
                tidy, x="date", y="interest", color="keyword",
                title="Google Trends по нашим темам в России (последние 12 месяцев)",
            )
            summary = self.result.tables["trends_summary"]
            self.result.figures["trends_avg"] = build_bar(
                summary, x="keyword", y="avg_interest",
                title="Средний поисковый интерес по темам (Google Trends)",
            )

            top = summary.iloc[0]
            self.result.add_finding(
                f"Самый горячий поисковый запрос в России - '{top['keyword']}' "
                f"(средний индекс {top['avg_interest']:.0f}, пик {top['peak_interest']:.0f}). "
                "Делаем посадочную страницу и контент-стратегию под него."
            )
            cooling = summary[summary["last_value"] < summary["avg_interest"] * 0.8]
            if not cooling.empty:
                names = ", ".join(cooling["keyword"].tolist())
                self.result.add_finding(
                    f"Темы со снижением интереса в последний месяц: {names}. "
                    "Их используем как поддерживающий контент, основной упор на растущие."
                )

        if "market_facts" in self.result.tables and not self.result.tables["market_facts"].empty:
            facts = self.result.tables["market_facts"]
            preview = facts["snippet"].iloc[0]
            self.result.add_finding(
                f"Wikipedia (статья 'Образовательные технологии') даёт публичные оценки рынка. "
                f"Например: '{preview}'. На эти цифры опираемся в инвестпрезентации."
            )

        self.result.add_finding(
            f"Средний чек заложен {self.average_check:.0f} руб., целевая маржа 55 процентов, "
            "значит максимальный CAC, который мы можем себе позволить, около 13 700 руб."
        )


In [78]:
marketing_result = MarketingResearch().run()
if "trends_summary" in marketing_result.tables:
    display(marketing_result.tables["trends_summary"].round(1))
if "market_facts" in marketing_result.tables:
    display(marketing_result.tables["market_facts"])

for figure in marketing_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in marketing_result.findings:
    print(" -", item)


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,keyword,avg_interest,peak_interest,last_value
3,уход за лицом,71.6,100,57
1,базовый гардероб,9.5,58,21
0,looksmaxxing,2.6,32,17
2,стилист онлайн,1.1,36,8


Выводы по разделу:
 - Самый горячий поисковый запрос в России - 'уход за лицом' (средний индекс 72, пик 100). Делаем посадочную страницу и контент-стратегию под него.
 - Темы со снижением интереса в последний месяц: уход за лицом. Их используем как поддерживающий контент, основной упор на растущие.
 - Средний чек заложен 24900 руб., целевая маржа 55 процентов, значит максимальный CAC, который мы можем себе позволить, около 13 700 руб.


## 9. Сводный дашборд на Dash


In [79]:
from dash import Dash, dash_table, dcc, html

PIPELINE_RESULTS = {
    "contractors": contractors_result,
    "geography": geography_result,
    "employees": employees_result,
    "marketing": marketing_result,
}

TABS = [
    ("Контрагенты", "contractors"),
    ("География", "geography"),
    ("Сотрудники", "employees"),
    ("Маркетинг", "marketing"),
]


def _table_card(title, df):
    df_show = df.copy()
    for col in df_show.columns:
        if df_show[col].dtype == "object":
            df_show[col] = df_show[col].astype(str).str.slice(0, 220)
    return html.Div([
        html.H4(title),
        dash_table.DataTable(
            data=df_show.round(2).to_dict("records"),
            columns=[{"name": c, "id": c} for c in df_show.columns],
            style_table={"overflowX": "auto"},
            style_cell={"fontFamily": "Inter, Arial, sans-serif",
                        "fontSize": "13px", "padding": "6px"},
            style_header={"backgroundColor": "#f4f6ff", "fontWeight": "600"},
            page_size=12,
        ),
    ], style={"marginBottom": "24px"})


def _findings_card(findings):
    items = [html.Li(text) for text in findings]
    return html.Div([
        html.H4("Выводы по разделу"),
        html.Ul(items, style={"lineHeight": "1.55"}),
    ], style={"backgroundColor": "#fafbff", "padding": "16px 20px",
              "borderRadius": "12px", "marginBottom": "24px",
              "border": "1px solid #e7ebff"})


def _figures_grid(figures):
    return html.Div(
        [html.Div(dcc.Graph(figure=fig),
                  style={"flex": "1 1 460px", "minWidth": "420px",
                         "marginBottom": "12px"})
         for fig in figures.values()],
        style={"display": "flex", "flexWrap": "wrap", "gap": "12px"},
    )


def build_layout():
    tabs_children = []
    for label, key in TABS:
        module_result = PIPELINE_RESULTS[key]
        tab_content = html.Div([
            _findings_card(module_result.findings),
            _figures_grid(module_result.figures),
            html.Div([_table_card(name, table)
                      for name, table in module_result.tables.items()]),
        ], style={"padding": "12px 4px"})
        tabs_children.append(dcc.Tab(label=label, value=key, children=tab_content))
    return html.Div([
        html.Header([
            html.H1("GlowUp Research: исследование рынка"),
            html.P("Реальные данные: Wikipedia + Nominatim + hh.ru API + Google Trends.",
                   style={"color": "#5a607a", "marginTop": 0}),
        ], style={"padding": "24px 28px", "borderBottom": "1px solid #eee"}),
        dcc.Tabs(id="tabs", value=TABS[0][1], children=tabs_children,
                 style={"margin": "0 16px"}),
    ], style={"fontFamily": "Inter, Arial, sans-serif",
              "backgroundColor": "#fff", "minHeight": "100vh"})


app = Dash(__name__, title="GlowUp Research (real data)")
app.layout = build_layout()
print("Dash приложение собрано.")


Dash приложение собрано.


In [80]:
import threading, time
from IPython.display import IFrame, display

def _run_app():
    app.run(port=8051, debug=False, use_reloader=False)

threading.Thread(target=_run_app, daemon=True).start()
time.sleep(3)

try:
    from google.colab.output import eval_js
    proxy_url = eval_js("google.colab.kernel.proxyPort(8051)")
    print("Ссылка на дашборд:", proxy_url)
    display(IFrame(src=proxy_url, width="100%", height=900))
except ImportError:
    display(IFrame(src="http://127.0.0.1:8051", width="100%", height=900))


Dash is running on http://127.0.0.1:8051/



INFO:dash.dash:Dash is running on http://127.0.0.1:8051/



 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 8051 is in use by another program. Either identify and stop that program, or start the server with a different port.


Ссылка на дашборд: https://8051-m-s-kkb-use1d1-bglicjscomwg-d.us-east1-1.prod.colab.dev
